# Silver Layer — Customers
## SalesFlow Data Lakehouse | Phase 4: Curated Layer

Reads `salesflow_dev.bronze.customers`, applies cleaning and standardization,
and writes the curated result to `salesflow_dev.silver.customers`.

**Transformations applied:**
| Step | Transformation |
|---|---|
| 1 | Remove duplicates by `CustomerID` |
| 2 | Clean and standardize `CompanyName` |
| 3 | Standardize `Country` and `City` |
| 4 | Standardize `Phone` |
| 5 | Fill nulls: `ContactName`, `Region`, `PostalCode` |
| 6 | Add `data_quality_status` flag |
| 7 | Add `processing_timestamp` |

In [0]:
# Import shared utility functions from the Utils notebook
%run /Workspace/Users/bruno.ogs@gmail.com/sales_data_lakehouse_ls_mentorship/04_Utils/common_functions.ipynb

## 1. Read from Bronze

In [0]:
from pyspark.sql.functions import current_timestamp

# Read customers table from Bronze layer
df = spark.table("salesflow_dev.bronze.customers")

print(f"Records read from Bronze: {df.count()}")
display(df.limit(5))

## 2. Remove Duplicates
Deduplicate by primary key `CustomerID`, keeping the most recently ingested record.

In [0]:
# Drop duplicate CustomerIDs — primary key must be unique in Silver
df = df.dropDuplicates(["CustomerID"])

print(f"Records after deduplication: {df.count()}")

## 3. Clean and Standardize Columns
Applies string cleaning, phone standardization, and null filling.

In [0]:
from pyspark.sql.functions import coalesce, lit

# 3.1 Clean CompanyName using shared utility
df = clean_string_column(df, "CompanyName")

# 3.2 Standardize Country and City
df = clean_string_column(df, "Country")
df = clean_string_column(df, "City")

# 3.3 Standardize Phone using shared utility
df = standardize_phone(df, "Phone")

# 3.4 Fill null values with defined defaults
df = (
    df
    .fillna({"ContactName": "Unknown"})   # unknown contact
    .fillna({"Region": "N/A"})            # region not always applicable
    .fillna({"PostalCode": "00000"})      # placeholder for missing postal codes
)

## 4. Add Quality Flag
Marks records as `VALID` or `INVALID` based on mandatory fields.  
A record is `INVALID` if `CompanyName` or `Country` is null.

In [0]:
# Add data quality flag — CompanyName and Country are mandatory
df = add_quality_flag(df, ["CompanyName", "Country"])

## 5. Add Processing Timestamp
Captures when this record was processed in the Silver layer.

In [0]:
# Add processing timestamp to track when Silver transformation ran
df = df.withColumn("processing_timestamp", current_timestamp())

## 6. Save as Delta Table

In [0]:
# Write to Silver layer as Delta table — overwrite for first load
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("salesflow_dev.silver.customers")

print("Table saved: salesflow_dev.silver.customers")

## 7. Validation

In [0]:
silver_customers = spark.table("salesflow_dev.silver.customers")

# Record count
print(f"Total records: {silver_customers.count()}")

# Quality flag distribution — how many VALID vs INVALID
print("\nQuality flag distribution:")
display(silver_customers.groupBy("data_quality_status").count())

# Schema
print("\nSchema:")
silver_customers.printSchema()

# Sample
print("\nFirst 5 rows:")
display(silver_customers.limit(5))